# ADULT INCOME DATASET — DEPLOYMENT NOTEBOOK

### Best Model: Random Forest Classifier
This notebook trains only the best model (Random Forest) and saves it as `model.pkl` for Streamlit deployment.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

print('Libraries imported successfully!')

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv('adult.csv')
print('Dataset Shape:', df.shape)
df.head()

## Step 3: Data Cleaning

In [ ]:
# Replace '?' with NaN
df.replace('?', None, inplace=True)

# Fill missing values with mode
df['workclass'].fillna(df['workclass'].mode()[0], inplace=True)
df['occupation'].fillna(df['occupation'].mode()[0], inplace=True)
df['native-country'].fillna(df['native-country'].mode()[0], inplace=True)

print('Missing values after cleaning:')
print(df.isnull().sum())

## Step 4: Outlier Removal

In [ ]:
numerical_cols = df.select_dtypes(include=['int64']).columns

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print('Shape after outlier removal:', df.shape)

## Step 5: Feature Engineering

In [ ]:
# Separate features and target
X = df.drop('income', axis=1)
y = df['income'].map({'<=50K': 0, '>50K': 1})

# Identify column types
numerical_features = X.select_dtypes(include=['int64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numerical features:', numerical_features)
print('Categorical features:', categorical_features)
print('Target distribution:\n', y.value_counts())

## Step 6: Build Preprocessing Pipeline

In [ ]:
# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Column Transformer
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print('Preprocessor built!')

## Step 7: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## Step 8: Train Best Model — Random Forest

In [ ]:
# Build Random Forest pipeline (best model)
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline_rf.fit(X_train, y_train)
print('Random Forest trained successfully!')

## Step 9: Evaluate Model

In [ ]:
train_pred = pipeline_rf.predict(X_train)
test_pred  = pipeline_rf.predict(X_test)

print('=== RANDOM FOREST (Best Model) ===')
print(f'Training Accuracy : {accuracy_score(y_train, train_pred):.4f}')
print(f'Testing  Accuracy : {accuracy_score(y_test, test_pred):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, test_pred, target_names=['<=50K', '>50K']))

## Step 10: Save Model for Deployment

In [ ]:
# Save the full pipeline (preprocessor + model)
joblib.dump(pipeline_rf, 'model1.pkl')
print('Model saved as model.pkl — ready for Streamlit deployment!')

## Step 11: Verify Saved Model

In [ ]:
# Load and test
loaded_model = joblib.load('model1.pkl')
sample = X_test.iloc[:3]
preds  = loaded_model.predict(sample)
labels = {0: '<=50K', 1: '>50K'}

print('Sample predictions from loaded model:')
for i, p in enumerate(preds):
    print(f'  Sample {i+1}: {labels[p]}')

print('\nModel is ready for deployment!')